# Lab 4: From CSV to SQL — Clinical Data Foundations (SOLUTIONS)
## CCI Session 3

**Duration:** 15 minutes

### Clinical Scenario
> KHCC's VistA system exports patient vital signs as CSV files daily. The vista_vitals dataset contains thousands of readings — blood pressure, pulse, pulse oximetry, respiration, and temperature — from multiple wards. You need to load this data, explore it, convert it to a SQL database, and answer clinical questions.

### Objective
- Load and explore clinical CSV data with pandas
- Convert a DataFrame to a SQLite database
- Write SQL queries for clinical analytics

---
### Setup

In [ ]:
# Setup
!pip install pandas -q

import pandas as pd
import sqlite3

print('Setup complete!')

---
## Cell 1 — Load the CSV

In [ ]:
# === CELL 1: LOAD CSV ===
import pandas as pd

# Load the CSV file
df = pd.read_csv('vista_vitals.csv')

# Print basic info
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)

# Display the first 10 rows
df.head(10)

---
## Cell 1b — Load Patient Demographics

> **Note:** The `NAME` and `ARABIC_NAME` columns contain Fernet-encrypted strings to protect patient identity.

In [ ]:
# === CELL 1b: LOAD PATIENT DEMOGRAPHICS ===

# Load vista_patients.csv into a second DataFrame
df_patients = pd.read_csv('vista_patients.csv')

# Print basic info: shape, columns, dtypes
print(f"Shape: {df_patients.shape}")
print(f"\nColumns: {list(df_patients.columns)}")
print(f"\nData types:")
print(df_patients.dtypes)

# Explore key demographic columns
print(f"\nSEX distribution:")
print(df_patients['SEX'].value_counts())

print(f"\nNATIONALITY distribution:")
print(df_patients['NATIONALITY'].value_counts())

print(f"\nPROVINCE distribution:")
print(df_patients['PROVINCE'].value_counts())

# Display the first 10 rows
df_patients.head(10)

---
## Cell 2 — Data Exploration

In [ ]:
# === CELL 2: DATA EXPLORATION ===

# How many unique patients (MRNs)?
print(f"Unique patients: {df['MRN'].nunique()}")

# What are the vital types and how many of each?
print(f"\nVital type counts:")
print(df['VITAL_TYPE'].value_counts())

# What hospital locations are represented?
print(f"\nHospital locations:")
print(df['HOSPITAL_LOCATION'].unique())

# What is the date range of the data?
print(f"\nDate range:")
print(f"  Earliest: {df['DATE_TIME_VITALS_TAKEN'].min()}")
print(f"  Latest:   {df['DATE_TIME_VITALS_TAKEN'].max()}")

# For PULSE readings only, show describe() statistics
df_pulse = df[df['VITAL_TYPE'] == 'PULSE'].copy()
df_pulse['RATE_NUM'] = pd.to_numeric(df_pulse['RATE'], errors='coerce')
print(f"\nPULSE statistics:")
print(df_pulse['RATE_NUM'].describe())

---
## Cell 3 — Convert to SQLite

In [ ]:
# === CELL 3: PANDAS TO SQLITE ===
import sqlite3

# Create a SQLite connection (file-based)
conn = sqlite3.connect('khcc_vitals.db')

# Write the DataFrame to a SQL table called 'vista_vitals'
df.to_sql('vista_vitals', conn, if_exists='replace', index=False)

# Verify by running a simple query
result = pd.read_sql("SELECT COUNT(*) as total_rows FROM vista_vitals", conn)
print(f"Total rows in database: {result['total_rows'][0]}")
print(f"Matches DataFrame: {result['total_rows'][0] == len(df)}")

---
## Cell 3b — Load Patient Demographics into SQLite

In [ ]:
# === CELL 3b: LOAD PATIENTS INTO SQLITE ===

# Load vista_patients into the same SQLite database
df_patients.to_sql('vista_patients', conn, if_exists='replace', index=False)

# Verify by running a simple count query
result = pd.read_sql("SELECT COUNT(*) as total_patients FROM vista_patients", conn)
print(f"Total patients in database: {result['total_patients'][0]}")

# Check that both tables exist
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"\nTables in database: {list(tables['name'])}")

---
## Cell 4 — Basic SQL Queries

In [ ]:
# === CELL 4: SQL QUERIES ===

# Query 1: Find all temperature readings above 100.4°F (fever)
print("=== Query 1: Fever readings ===")
q1 = pd.read_sql("""
    SELECT MRN, RATE, HOSPITAL_LOCATION, DATE_TIME_VITALS_TAKEN
    FROM vista_vitals
    WHERE VITAL_TYPE = 'TEMPERATURE'
      AND CAST(RATE AS FLOAT) > 100.4
""", conn)
print(q1)

# Query 2: Count readings per hospital location
print("\n=== Query 2: Readings per location ===")
q2 = pd.read_sql("""
    SELECT HOSPITAL_LOCATION, COUNT(*) as reading_count
    FROM vista_vitals
    GROUP BY HOSPITAL_LOCATION
    ORDER BY reading_count DESC
""", conn)
print(q2)

# Query 3: Find all vitals for a specific patient MRN
# First, get a sample MRN
sample_mrn = df['MRN'].iloc[0]
print(f"\n=== Query 3: Vitals for patient {sample_mrn} ===")
q3 = pd.read_sql(f"""
    SELECT VITAL_TYPE, RATE, DATE_TIME_VITALS_TAKEN, HOSPITAL_LOCATION
    FROM vista_vitals
    WHERE MRN = '{sample_mrn}'
    ORDER BY DATE_TIME_VITALS_TAKEN
""", conn)
print(q3)

# Query 4: Count unique patients per ward
print("\n=== Query 4: Unique patients per ward ===")
q4 = pd.read_sql("""
    SELECT HOSPITAL_LOCATION, COUNT(DISTINCT MRN) as unique_patients
    FROM vista_vitals
    GROUP BY HOSPITAL_LOCATION
    ORDER BY unique_patients DESC
""", conn)
print(q4)

# Query 5: Find the most recent vitals for each vital type
print("\n=== Query 5: Most recent vital per type ===")
q5 = pd.read_sql("""
    SELECT VITAL_TYPE, MRN, RATE, MAX(DATE_TIME_VITALS_TAKEN) as latest_reading
    FROM vista_vitals
    GROUP BY VITAL_TYPE
""", conn)
print(q5)

---
## Cell 5 — Advanced SQL Queries

In [ ]:
# === CELL 5: ADVANCED QUERIES ===

# Query 6: Average pulse rate by ward
print("=== Query 6: Average pulse by ward ===")
q6 = pd.read_sql("""
    SELECT HOSPITAL_LOCATION,
           ROUND(AVG(CAST(RATE AS FLOAT)), 1) as avg_pulse
    FROM vista_vitals
    WHERE VITAL_TYPE = 'PULSE'
    GROUP BY HOSPITAL_LOCATION
    ORDER BY avg_pulse DESC
""", conn)
print(q6)

# Query 7: Patients who had BOTH fever AND tachycardia
print("\n=== Query 7: Fever + Tachycardia patients ===")
q7 = pd.read_sql("""
    SELECT DISTINCT f.MRN, f.RATE as fever_temp, t.RATE as pulse_rate, f.HOSPITAL_LOCATION
    FROM vista_vitals f
    JOIN vista_vitals t ON f.MRN = t.MRN
    WHERE f.VITAL_TYPE = 'TEMPERATURE'
      AND CAST(f.RATE AS FLOAT) > 100.4
      AND t.VITAL_TYPE = 'PULSE'
      AND CAST(t.RATE AS FLOAT) > 100
""", conn)
print(q7)

# Query 8: Vital readings per hour of day
print("\n=== Query 8: Readings per hour ===")
q8 = pd.read_sql("""
    SELECT CAST(SUBSTR(DATE_TIME_VITALS_TAKEN, 12, 2) AS INTEGER) as hour_of_day,
           COUNT(*) as reading_count
    FROM vista_vitals
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""", conn)
print(q8)

---
## Cell 6 — The Blood Pressure Challenge

In [ ]:
# === CELL 6: BLOOD PRESSURE PARSING ===

# Create a new DataFrame with BP readings only
df_bp = df[df['VITAL_TYPE'] == 'BLOOD PRESSURE'].copy()
print(f"Blood pressure readings: {len(df_bp)}")

# Split the RATE column into systolic and diastolic
df_bp['systolic'] = df_bp['RATE'].str.split('/').str[0].astype(int)
df_bp['diastolic'] = df_bp['RATE'].str.split('/').str[1].astype(int)

print(f"\nBP Statistics:")
print(df_bp[['systolic', 'diastolic']].describe())

# Find patients with hypertensive readings (systolic > 140)
hypertensive = df_bp[df_bp['systolic'] > 140]
print(f"\nHypertensive readings (systolic > 140): {len(hypertensive)}")
print(hypertensive[['MRN', 'RATE', 'systolic', 'diastolic', 'HOSPITAL_LOCATION']].head(10))

# Find patients with hypotension (systolic < 90)
hypotensive = df_bp[df_bp['systolic'] < 90]
print(f"\nHypotensive readings (systolic < 90): {len(hypotensive)}")
print(hypotensive[['MRN', 'RATE', 'systolic', 'diastolic', 'HOSPITAL_LOCATION']].head(10))

---
## Cell 5b — JOIN Queries: Combining Vitals with Patient Demographics

Now that you have **two tables** in your database (`vista_vitals` and `vista_patients`), you can use SQL JOINs to combine them. The **MRN** column is the join key that links vitals to patient demographics.

In [ ]:
# === CELL 5b: JOIN QUERIES ===

# JOIN Query 1: Find all female patients who had fever (temperature > 100.4)
print("=== JOIN Query 1: Female patients with fever ===")
jq1 = pd.read_sql("""
    SELECT DISTINCT p.MRN, p.SEX, p.DOB, p.NATIONALITY, v.RATE, v.DATE_TIME_VITALS_TAKEN
    FROM vista_vitals v
    JOIN vista_patients p ON v.MRN = p.MRN
    WHERE v.VITAL_TYPE = 'TEMPERATURE'
      AND CAST(v.RATE AS FLOAT) > 100.4
      AND p.SEX = 'FEMALE'
""", conn)
print(jq1)

# JOIN Query 2: Average pulse rate by patient sex
print("\n=== JOIN Query 2: Average pulse rate by sex ===")
jq2 = pd.read_sql("""
    SELECT p.SEX, 
           COUNT(*) as reading_count,
           ROUND(AVG(CAST(v.RATE AS FLOAT)), 1) as avg_pulse
    FROM vista_vitals v
    JOIN vista_patients p ON v.MRN = p.MRN
    WHERE v.VITAL_TYPE = 'PULSE'
    GROUP BY p.SEX
    ORDER BY avg_pulse DESC
""", conn)
print(jq2)

# JOIN Query 3: Count vital readings by nationality
print("\n=== JOIN Query 3: Vital readings by nationality ===")
jq3 = pd.read_sql("""
    SELECT p.NATIONALITY, 
           COUNT(*) as total_readings,
           COUNT(DISTINCT v.MRN) as unique_patients
    FROM vista_vitals v
    JOIN vista_patients p ON v.MRN = p.MRN
    GROUP BY p.NATIONALITY
    ORDER BY total_readings DESC
""", conn)
print(jq3)

# JOIN Query 4: Find patients from AMMAN with abnormal blood pressure (systolic > 140)
print("\n=== JOIN Query 4: AMMAN patients with high blood pressure ===")
jq4 = pd.read_sql("""
    SELECT DISTINCT p.MRN, p.SEX, p.DOB, p.PROVINCE, v.RATE, v.DATE_TIME_VITALS_TAKEN
    FROM vista_vitals v
    JOIN vista_patients p ON v.MRN = p.MRN
    WHERE v.VITAL_TYPE = 'BLOOD PRESSURE'
      AND CAST(SUBSTR(v.RATE, 1, INSTR(v.RATE, '/') - 1) AS INTEGER) > 140
      AND p.PROVINCE = 'AMMAN'
""", conn)
print(jq4)

# JOIN Query 5: List all vitals for the oldest patient in the database
print("\n=== JOIN Query 5: Vitals for the oldest patient ===")
jq5 = pd.read_sql("""
    SELECT p.MRN, p.DOB, p.SEX, p.NATIONALITY, 
           v.VITAL_TYPE, v.RATE, v.DATE_TIME_VITALS_TAKEN, v.HOSPITAL_LOCATION
    FROM vista_vitals v
    JOIN vista_patients p ON v.MRN = p.MRN
    WHERE p.MRN = (
        SELECT MRN FROM vista_patients 
        WHERE DOB IS NOT NULL AND DOB != ''
        ORDER BY DOB ASC 
        LIMIT 1
    )
    ORDER BY v.DATE_TIME_VITALS_TAKEN
""", conn)
print(jq5)

---
## Stretch Challenge — Clinical Alert Report

In [ ]:
# === STRETCH CHALLENGE: CLINICAL ALERT REPORT ===

# Identify critical vital signs
# Fever: temperature > 100.4
fever = df[df['VITAL_TYPE'] == 'TEMPERATURE'].copy()
fever['RATE_NUM'] = pd.to_numeric(fever['RATE'], errors='coerce')
fever = fever[fever['RATE_NUM'] > 100.4]
fever['alert'] = fever.apply(lambda r: f"Fever ({r['RATE']}°F)", axis=1)

# Tachycardia: pulse > 100
tachy = df[df['VITAL_TYPE'] == 'PULSE'].copy()
tachy['RATE_NUM'] = pd.to_numeric(tachy['RATE'], errors='coerce')
tachy = tachy[tachy['RATE_NUM'] > 100]
tachy['alert'] = tachy.apply(lambda r: f"Tachycardia ({r['RATE']} bpm)", axis=1)

# Hypotension: systolic < 90
hypo = df_bp[df_bp['systolic'] < 90].copy()
hypo['alert'] = hypo.apply(lambda r: f"Hypotension ({r['RATE']})", axis=1)

# Low SpO2: < 92%
low_spo2 = df[df['VITAL_TYPE'] == 'PULSE OXIMETRY'].copy()
low_spo2['RATE_NUM'] = pd.to_numeric(low_spo2['RATE'], errors='coerce')
low_spo2 = low_spo2[low_spo2['RATE_NUM'] < 92]
low_spo2['alert'] = low_spo2.apply(lambda r: f"Low SpO2 ({r['RATE']}%)", axis=1)

# Combine all alerts
all_alerts = pd.concat([
    fever[['MRN', 'HOSPITAL_LOCATION', 'alert']],
    tachy[['MRN', 'HOSPITAL_LOCATION', 'alert']],
    hypo[['MRN', 'HOSPITAL_LOCATION', 'alert']],
    low_spo2[['MRN', 'HOSPITAL_LOCATION', 'alert']]
])

# Print the report
print("CLINICAL ALERT REPORT")
print("=" * 50)

if len(all_alerts) == 0:
    print("No critical alerts found.")
else:
    for ward, ward_data in all_alerts.groupby('HOSPITAL_LOCATION'):
        print(f"\nWard: {ward}")
        for mrn, patient_data in ward_data.groupby('MRN'):
            alerts = ', '.join(patient_data['alert'].values)
            print(f"  - Patient {mrn}: {alerts}")

print(f"\nTotal alerts: {len(all_alerts)}")
print(f"Patients affected: {all_alerts['MRN'].nunique()}")

---
### KHCC Connection

This lab mirrors how the **AIDI team** processes daily VistA extracts at KHCC.
Every day, vital signs data flows from the VistA system as flat files, gets loaded into
databases, and is queried for clinical analytics and dashboard reporting. The skills you
practiced here — loading CSVs, converting to SQL, writing queries — are the exact
foundation of clinical data engineering.